# CogFlex Suite Benchmark

Kaggle runtime notebook for the cognitive-flexibility suite.

In [ ]:
import os
from pathlib import Path

DATASET_ROOT_ENV_VAR = "RULESHIFT_DATASET_ROOT"
PRIVATE_DATASET_ROOT_ENV_VAR = "RULESHIFT_PRIVATE_DATASET_ROOT"
PRIVATE_ANSWER_KEY_PATH_ENV_VAR = "RULESHIFT_PRIVATE_ANSWER_KEY_PATH"
EVAL_SPLIT = os.environ.get("RULESHIFT_EVAL_SPLIT", "public").strip().lower()
PUBLIC_ROWS_FILENAME = "public_leaderboard_rows.json"
PRIVATE_ROWS_FILENAME = "private_leaderboard_rows.json"
DEFAULT_DATASET_ROOT = Path("/kaggle/input/datasets/raptorengineer/cogflex-suite-runtime")
DEFAULT_PRIVATE_DATASET_ROOT = Path("/kaggle/input/datasets/raptorengineer/cogflex-suite-runtime-private")
DATASET_ROOT = Path(os.environ.get(DATASET_ROOT_ENV_VAR, str(DEFAULT_DATASET_ROOT)).strip() or DEFAULT_DATASET_ROOT)
PRIVATE_DATASET_ROOT = Path(
    os.environ.get(PRIVATE_DATASET_ROOT_ENV_VAR, str(DEFAULT_PRIVATE_DATASET_ROOT)).strip() or DEFAULT_PRIVATE_DATASET_ROOT
)
ROWS_PATH = DATASET_ROOT / PUBLIC_ROWS_FILENAME if EVAL_SPLIT == "public" else PRIVATE_DATASET_ROOT / PRIVATE_ROWS_FILENAME
_PRIVATE_ANSWER_KEY_PATH = os.environ.get(PRIVATE_ANSWER_KEY_PATH_ENV_VAR, "").strip()
PRIVATE_ANSWER_KEY_PATH = Path(_PRIVATE_ANSWER_KEY_PATH) if _PRIVATE_ANSWER_KEY_PATH else None


In [ ]:
from dataclasses import dataclass
from enum import Enum
import re

PROBE_COUNT = 4
TURN_COUNT = 3
PROBE_FIELDS = ("probe_1", "probe_2", "probe_3", "probe_4")
TURN_HEADER_PREFIX = "CogFlex suite task. Episode "
OUTPUT_INSTRUCTION = "Return exactly 4 outputs in order, one per probe. Use only type_a or type_b."
FACULTY_ID = "executive_functions/cognitive_flexibility"
ALLOWED_SUITE_TASKS = {
    "explicit_rule_update",
    "latent_rule_update",
    "context_binding",
    "trial_cued_switch",
}
SUITE_SHIFT_MODES = {
    "explicit_rule_update": "explicit_instruction",
    "latent_rule_update": "latent_example_change",
    "context_binding": "context_gate",
    "trial_cued_switch": "cue_switching",
}
EXPECTED_PUBLIC_EPISODE_COUNT = 80
EXPECTED_PRIVATE_EPISODE_COUNT = 400
EXPECTED_EPISODES_PER_TASK = {"public": 20, "private": 100}
LINE_RE = re.compile(r"^(?P<index>\d+)\.\s+(?P<body>.+?)\s+->\s+(?P<label>type_a|type_b|\?)$")
POINT_RE = re.compile(r"^r1=(?P<r1>[+-]\d+),\s*r2=(?P<r2>[+-]\d+)$")
_ALLOWED_LABELS = {"type_a", "type_b"}

class Label(str, Enum):
    TYPE_A = "type_a"
    TYPE_B = "type_b"

@dataclass(frozen=True)
class BinaryResponse:
    probe_1: Label
    probe_2: Label
    probe_3: Label
    probe_4: Label

    def as_tuple(self) -> tuple[str, str, str, str]:
        return (
            _coerce_label(self.probe_1, "probe_1"),
            _coerce_label(self.probe_2, "probe_2"),
            _coerce_label(self.probe_3, "probe_3"),
            _coerce_label(self.probe_4, "probe_4"),
        )

class KaggleExecutionError(RuntimeError):
    pass


In [ ]:
def normalize_binary_response(response: object) -> tuple[str, ...] | None:
    if response is None:
        return None
    if isinstance(response, BinaryResponse):
        return response.as_tuple()
    if isinstance(response, str):
        return _parse_text_response(response)
    coerced = _try_coerce(response)
    return coerced.as_tuple() if coerced is not None else None

def _parse_text_response(text: str) -> tuple[str, ...] | None:
    tokens = tuple(
        item.strip().lower()
        for item in text.strip().strip("`").replace("\n", ",").split(",")
        if item.strip()
    )
    return _norm_labels(tokens)

def _try_coerce(response: object) -> BinaryResponse | None:
    if isinstance(response, dict):
        values = response
    elif hasattr(response, "__getitem__") and hasattr(response, "keys"):
        try:
            values = {field: response[field] for field in PROBE_FIELDS}
        except (KeyError, TypeError):
            return None
    elif all(hasattr(response, field) for field in PROBE_FIELDS):
        values = {field: getattr(response, field) for field in PROBE_FIELDS}
    else:
        return None
    try:
        labels = tuple(Label(_coerce_label(values[field], field)) for field in PROBE_FIELDS)
    except (KeyError, TypeError):
        return None
    return BinaryResponse(*labels)

def _coerce_label(value: object, field: str) -> str:
    try:
        return _normalize_label(value)
    except ValueError as exc:
        raise ValueError(f"invalid field {field}: {value!r}") from exc

def _norm_labels(labels: tuple[str, ...] | tuple[Label, ...] | None) -> tuple[str, ...] | None:
    if labels is None:
        return None
    try:
        normalized = tuple(_normalize_label(label.value if isinstance(label, Label) else label) for label in labels)
    except ValueError:
        return None
    return normalized if len(normalized) == PROBE_COUNT else None

def _normalize_label(value: object) -> str:
    if isinstance(value, Enum):
        value = value.value
    elif hasattr(value, "value"):
        value = value.value
    normalized = str(value).strip().lower()
    if normalized not in _ALLOWED_LABELS:
        raise ValueError(f"unknown label: {value}")
    return normalized


In [ ]:
def score_episode(
    predictions: tuple[str, ...] | tuple[Label, ...] | None,
    targets: tuple[str, ...] | tuple[Label, ...],
) -> dict:
    norm_targets = _norm_labels(targets)
    if norm_targets is None:
        raise ValueError(f"targets must contain exactly {PROBE_COUNT} valid labels")
    norm_predictions = _norm_labels(predictions)
    if norm_predictions is None:
        norm_predictions = tuple("" for _ in range(PROBE_COUNT))
        numerator = 0
    else:
        numerator = sum(pred == target for pred, target in zip(norm_predictions, norm_targets))
    return {
        "numerator": numerator,
        "denominator": PROBE_COUNT,
        "predictions": list(norm_predictions),
    }

@kbench.task(store_task=False)
def run_binary_task(
    llm: object,
    turns: list[str],
    final_probe_targets: tuple[str, ...] | tuple[Label, ...],
) -> dict:
    if len(turns) != TURN_COUNT:
        raise KaggleExecutionError(f"expected {TURN_COUNT} turns, received {len(turns)}")
    try:
        llm.prompt(turns[0])
        llm.prompt(turns[1])
        response = llm.prompt(turns[2], schema=BinaryResponse)
    except Exception as exc:
        raise KaggleExecutionError("llm.prompt failed") from exc

    try:
        normalized = normalize_binary_response(response)
    except ValueError as exc:
        raise KaggleExecutionError(f"invalid binary response: {exc}") from exc

    if normalized is None:
        raise KaggleExecutionError(f"unscoreable response of type {type(response).__name__}")
    return score_episode(normalized, final_probe_targets)


In [ ]:
import json
from collections import Counter

def load_selected_rows() -> list[dict[str, object]]:
    return _load_rows(ROWS_PATH)

def attach_selected_scoring(rows: list[dict[str, object]]) -> list[dict[str, object]]:
    if EVAL_SPLIT == "public":
        return rows
    return _attach_private_scoring(rows)

def _parse_case_line(line: str) -> dict[str, object] | None:
    match = LINE_RE.match(line.strip())
    if match is None:
        return None
    body = match.group("body")
    attributes: dict[str, object] = {"index": int(match.group("index")), "label": match.group("label")}
    point = None
    for chunk in (part.strip() for part in body.split("|")):
        point_match = POINT_RE.match(chunk)
        if point_match is not None:
            point = (int(point_match.group("r1")), int(point_match.group("r2")))
            continue
        if "=" not in chunk:
            raise ValueError(f"malformed chunk: {chunk!r}")
        key, value = chunk.split("=", 1)
        attributes[key.strip()] = value.strip()
    if point is None:
        raise ValueError(f"missing coordinates in line: {line!r}")
    attributes["r1"] = point[0]
    attributes["r2"] = point[1]
    return attributes

def _count_turn_examples(turn: str) -> int:
    return sum(
        (parsed := _parse_case_line(line)) is not None and parsed["label"] != "?"
        for line in turn.splitlines()
    )

def _count_turn_probes(turn: str) -> int:
    return sum(
        (parsed := _parse_case_line(line)) is not None and parsed["label"] == "?"
        for line in turn.splitlines()
    )

def _validate_turn(turn: str, episode_id: str, turn_index: int) -> None:
    if not turn.startswith(f"{TURN_HEADER_PREFIX}{episode_id}. Turn {turn_index} of 3."):
        raise ValueError(f"episode {episode_id}: malformed header for turn {turn_index}")
    if turn_index in {1, 2}:
        if _count_turn_examples(turn) != 4:
            raise ValueError(f"episode {episode_id}: turn {turn_index} must contain 4 labeled examples")
    else:
        if _count_turn_probes(turn) != PROBE_COUNT:
            raise ValueError(f"episode {episode_id}: decision turn must contain {PROBE_COUNT} probes")
        if OUTPUT_INSTRUCTION not in turn:
            raise ValueError(f"episode {episode_id}: decision turn must use the fixed output instruction")

def _validate_row(row: dict[str, object]) -> None:
    episode_id = str(row["episode_id"])
    analysis = row["analysis"]
    turns = row["inference"]["turns"]

    if analysis["faculty_id"] != FACULTY_ID:
        raise ValueError(f"episode {episode_id}: unsupported faculty_id {analysis['faculty_id']!r}")
    if analysis["suite_task_id"] not in ALLOWED_SUITE_TASKS:
        raise ValueError(f"episode {episode_id}: unsupported suite_task_id {analysis['suite_task_id']!r}")
    if analysis["shift_mode"] != SUITE_SHIFT_MODES[analysis["suite_task_id"]]:
        raise ValueError(f"episode {episode_id}: shift_mode mismatch for suite task {analysis['suite_task_id']!r}")
    if analysis["difficulty_bin"] not in {"hard", "medium"}:
        raise ValueError(f"episode {episode_id}: unsupported difficulty_bin {analysis['difficulty_bin']!r}")
    if not isinstance(turns, list) or len(turns) != TURN_COUNT:
        raise ValueError(f"episode {episode_id}: expected exactly {TURN_COUNT} turns")
    for turn_index, turn in enumerate(turns, start=1):
        if not isinstance(turn, str) or not turn.strip():
            raise ValueError(f"episode {episode_id}: turn {turn_index} must be a non-empty string")
        _validate_turn(turn, episode_id, turn_index)

def _validate_batch(rows: list[dict[str, object]]) -> None:
    expected_total = EXPECTED_PUBLIC_EPISODE_COUNT if EVAL_SPLIT == "public" else EXPECTED_PRIVATE_EPISODE_COUNT
    if len(rows) != expected_total:
        raise RuntimeError(
            f"benchmark dataset mismatch: expected {expected_total} episodes for split {EVAL_SPLIT!r}, but loaded {len(rows)} from {ROWS_PATH}."
        )
    counts = Counter(str(row["analysis"]["suite_task_id"]) for row in rows)
    expected_per_task = EXPECTED_EPISODES_PER_TASK[EVAL_SPLIT]
    if set(counts) != ALLOWED_SUITE_TASKS:
        raise RuntimeError(f"benchmark suite tasks mismatch: expected {sorted(ALLOWED_SUITE_TASKS)}, got {sorted(counts)}")
    bad_counts = {suite_task_id: count for suite_task_id, count in counts.items() if count != expected_per_task}
    if bad_counts:
        raise RuntimeError(f"benchmark per-task counts mismatch for split {EVAL_SPLIT!r}: {bad_counts}")

def _normalize_final_probe_targets(values: object) -> tuple[str, ...]:
    if not isinstance(values, (list, tuple)):
        raise ValueError("final_probe_targets must be a list or tuple")
    labels = _norm_labels(tuple(values))
    if labels is None:
        raise ValueError(f"final_probe_targets must contain exactly {PROBE_COUNT} valid labels")
    return labels

def _load_rows(path: Path) -> list[dict[str, object]]:
    rows = json.loads(path.read_text("utf-8"))
    scoped_rows: list[dict[str, object]] = []
    for row in rows:
        analysis = row.get("analysis")
        inference = row.get("inference")
        scoring = row.get("scoring")
        if not isinstance(analysis, dict):
            raise RuntimeError(f"row {row.get('episode_id')} is missing analysis metadata")
        if not isinstance(inference, dict):
            raise RuntimeError(f"row {row.get('episode_id')} is missing inference fields")
        if EVAL_SPLIT == "public":
            if not isinstance(scoring, dict):
                raise RuntimeError(f"row {row.get('episode_id')} is missing scoring fields")
        elif scoring is not None:
            raise RuntimeError(f"row {row.get('episode_id')} leaks scoring fields in the private split")

        normalized_row = {
            "episode_id": row["episode_id"],
            "inference": {"turns": list(inference["turns"])},
            "analysis": {
                "faculty_id": analysis["faculty_id"],
                "suite_task_id": analysis["suite_task_id"],
                "shift_mode": analysis["shift_mode"],
                "difficulty_bin": analysis["difficulty_bin"],
            },
        }
        if EVAL_SPLIT == "public":
            normalized_row["scoring"] = {"final_probe_targets": _normalize_final_probe_targets(scoring["final_probe_targets"])}
        _validate_row(normalized_row)
        scoped_rows.append(normalized_row)

    _validate_batch(scoped_rows)
    return scoped_rows

def _load_private_answer_key(path: Path) -> dict[str, dict[str, object]]:
    payload = json.loads(path.read_text("utf-8"))
    if not isinstance(payload, dict):
        raise RuntimeError("private answer key payload must be a JSON object")
    if payload.get("version") != "cogflex_suite_v1":
        raise RuntimeError("private answer key has an unsupported version")
    if payload.get("split") != "private":
        raise RuntimeError("private answer key must declare split='private'")
    episodes = payload.get("episodes")
    if not isinstance(episodes, list):
        raise RuntimeError("private answer key must expose an episodes list")
    answer_map: dict[str, dict[str, object]] = {}
    for answer in episodes:
        if not isinstance(answer, dict):
            raise RuntimeError("private answer key episodes must be objects")
        episode_id = str(answer.get("episode_id"))
        if episode_id in answer_map:
            raise RuntimeError(f"private answer key duplicates episode_id {episode_id}")
        answer_map[episode_id] = answer
    return answer_map

def _attach_private_scoring(rows: list[dict[str, object]]) -> list[dict[str, object]]:
    if PRIVATE_ANSWER_KEY_PATH is None:
        raise RuntimeError(
            "Private split requires an external answer key. "
            f"Set {PRIVATE_ANSWER_KEY_PATH_ENV_VAR} to a local private_answer_key.json path."
        )
    if not PRIVATE_ANSWER_KEY_PATH.exists():
        raise FileNotFoundError(f"Private answer key not found: {PRIVATE_ANSWER_KEY_PATH}")
    answer_map = _load_private_answer_key(PRIVATE_ANSWER_KEY_PATH)
    scored_rows: list[dict[str, object]] = []
    seen_ids: set[str] = set()
    for row in rows:
        episode_id = str(row["episode_id"])
        answer = answer_map.get(episode_id)
        if answer is None:
            raise RuntimeError(f"private answer key is missing episode_id {episode_id}")
        for key in ("faculty_id", "suite_task_id", "shift_mode", "difficulty_bin"):
            if answer.get(key) != row["analysis"][key]:
                raise RuntimeError(f"private answer key {key} mismatch for episode {episode_id}")
        scored_rows.append(
            {
                "episode_id": row["episode_id"],
                "inference": {"turns": list(row["inference"]["turns"])},
                "scoring": {"final_probe_targets": _normalize_final_probe_targets(answer.get("final_probe_targets"))},
                "analysis": dict(row["analysis"]),
            }
        )
        seen_ids.add(episode_id)
    extra_ids = sorted(set(answer_map) - seen_ids)
    if extra_ids:
        raise RuntimeError(f"private answer key contains unknown episode_ids: {extra_ids}")
    return scored_rows

leaderboard_rows = load_selected_rows()
scored_rows = attach_selected_scoring(leaderboard_rows)
loaded_episode_count = len(scored_rows)
df = pd.DataFrame(
    [
        {
            "turns": row["inference"]["turns"],
            "final_probe_targets": row["scoring"]["final_probe_targets"],
        }
        for row in scored_rows
    ]
)


In [ ]:
def summarize_suite_benchmark(runs, rows: list[dict[str, object]]) -> dict[str, object]:
    if not runs:
        raise RuntimeError("evaluation produced no successful runs")
    results_df = runs.as_dataframe().reset_index(drop=True)
    if len(results_df) != len(rows):
        raise RuntimeError(f"evaluation completed {len(results_df)} of {len(rows)} benchmark episodes")

    overall_numerator = 0
    overall_denominator = 0
    per_task: dict[str, dict[str, int]] = {}
    per_shift_mode: dict[str, dict[str, int]] = {}
    per_difficulty: dict[str, dict[str, int]] = {}
    for row, result in zip(rows, results_df.result):
        suite_task_id = row["analysis"]["suite_task_id"]
        shift_mode = row["analysis"]["shift_mode"]
        difficulty_bin = row["analysis"]["difficulty_bin"]
        numerator = int(result["numerator"])
        denominator = int(result["denominator"])
        overall_numerator += numerator
        overall_denominator += denominator
        per_task.setdefault(suite_task_id, {"numerator": 0, "denominator": 0})
        per_shift_mode.setdefault(shift_mode, {"numerator": 0, "denominator": 0})
        per_difficulty.setdefault(difficulty_bin, {"numerator": 0, "denominator": 0})
        per_task[suite_task_id]["numerator"] += numerator
        per_task[suite_task_id]["denominator"] += denominator
        per_shift_mode[shift_mode]["numerator"] += numerator
        per_shift_mode[shift_mode]["denominator"] += denominator
        per_difficulty[difficulty_bin]["numerator"] += numerator
        per_difficulty[difficulty_bin]["denominator"] += denominator

    if overall_denominator == 0:
        raise RuntimeError("evaluation produced a zero denominator")

    per_task_accuracy = {
        suite_task_id: stats["numerator"] / stats["denominator"]
        for suite_task_id, stats in sorted(per_task.items())
    }
    macro_accuracy = sum(per_task_accuracy.values()) / len(per_task_accuracy)
    return {
        "score": macro_accuracy,
        "macro_accuracy": macro_accuracy,
        "micro_accuracy": overall_numerator / overall_denominator,
        "numerator": overall_numerator,
        "denominator": overall_denominator,
        "episodes": len(rows),
        "per_task_accuracy": per_task_accuracy,
        "shift_mode_accuracy": {
            key: value["numerator"] / value["denominator"] for key, value in sorted(per_shift_mode.items())
        },
        "difficulty_bin_accuracy": {
            key: value["numerator"] / value["denominator"] for key, value in sorted(per_difficulty.items())
        },
    }

@kbench.task(
    name="cogflex_suite_binary",
    description="Multi-task cognitive-flexibility suite with explicit, latent, context-bound, and cue-selected rule switching.",
)
def cogflex_suite_binary(llm) -> float:
    runs = run_binary_task.evaluate(llm=[llm], evaluation_data=df)
    summary = summarize_suite_benchmark(runs, scored_rows)
    print(json.dumps(summary, indent=2))
    return summary["score"]


In [ ]:
%choose cogflex_suite_binary
